# JSON справочника дашбордов → Excel

Просто выполните ячейки по порядку. В последней ячейке укажите путь к своему JSON-файлу.

In [ ]:
import json
from pathlib import Path

import pandas as pd


def as_list(value):
    """Приводит значение к списку: None -> [], словарь -> [словарь], список -> как есть."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def get_widgets(screen: dict):
    """Виджеты экрана могут лежать под ключом 'widget' или 'widgets', как список или как один объект."""
    for key in ("widget", "widgets"):
        if key in screen:
            return as_list(screen[key])
    return []


def get_screens(dashboard: dict):
    """Экраны дашборда - под ключом 'screens' (или 'screen')."""
    for key in ("screens", "screen"):
        if key in dashboard:
            return as_list(dashboard[key])
    return []


def get_dashboards(data):
    """Верхний уровень - под ключом 'dashboards', либо сам JSON уже список дашбордов."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("dashboards", "dashboard"):
            if key in data:
                return as_list(data[key])
    return []


def extract_rows(data):
    rows = []
    dashboards = get_dashboards(data)

    if not dashboards:
        print("⚠️  Не нашёл ключ 'dashboards' на верхнем уровне JSON - проверьте структуру файла.")
        return rows

    for dash in dashboards:
        if not isinstance(dash, dict):
            continue
        dash_id = dash.get("id")
        dash_name = dash.get("name")

        screens = get_screens(dash)
        if not screens:
            continue

        for screen in screens:
            if not isinstance(screen, dict):
                continue
            widgets = get_widgets(screen)
            for widget in widgets:
                if not isinstance(widget, dict):
                    continue
                rows.append({
                    "dashboard_id": dash_id,
                    "dashboard_name": dash_name,
                    "widget_id": widget.get("id"),
                    "widget_name": widget.get("name"),
                })

    return rows

In [ ]:
# Укажите путь к вашему JSON файлу:
JSON_PATH = Path("dashboards.json")
OUTPUT_PATH = JSON_PATH.with_suffix(".xlsx")

print(f"Читаю: {JSON_PATH.resolve()}")
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = extract_rows(data)
print(f"Найдено строк (dashboard x widget): {len(rows)}")

df = pd.DataFrame(rows, columns=["dashboard_id", "dashboard_name", "widget_id", "widget_name"])
df.to_excel(OUTPUT_PATH, index=False)
print(f"Сохранено: {OUTPUT_PATH.resolve()}")

df.head(20)